In [ ]:
# --- SPACEMAP notebook configuration ------------------------------------
# Every path below is built from DATA_DIR. Point it at your own data by
# setting SPACEMAP_DATA_DIR in your environment, or by editing the default.
import os
from pathlib import Path

DATA_DIR = os.environ.get("SPACEMAP_DATA_DIR", str(Path.home() / "spacemap-data"))
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Using DATA_DIR = {DATA_DIR}")


# Step 6E: Vortex (X-shift) Phenotyping

This notebook performs cell phenotyping using the Vortex/X-shift density-gradient clustering method on marker-expression data. It follows the same SPACEMAP phenotyping flow as the Leiden and SOM notebooks while using a distinct clustering strategy based on local density in k-nearest-neighbor space.

## Overview

The Vortex workflow:
1. **Data Preparation** - Load corrected or raw marker data
2. **Normalization** - Use the shared normalization cache for consistency
3. **Vortex Clustering** - Optional K diagnostics, X-shift clustering, and KNN backfilling
4. **Cluster Characterization** - Generate cluster summaries and expression heatmaps
5. **Manual Annotation** - Interactive phenotype assignment using the standard SPACEMAP studio
6. **Quality Control** - UMAP and cluster-size diagnostics for validation

## Input
- Batch-corrected or raw CSV data from earlier pipeline steps
- Corrected markers by default, with optional raw-marker mode
- Optional phenotype mapping for reproducible scripted assignments

## Output
- CSV files with Vortex cluster assignments
- CSV files with phenotype annotations
- K-selection diagnostics and cluster heatmaps
- QC figures including UMAP summaries when available

## Key Features
- **Different clustering method** - X-shift/Vortex density-gradient clustering instead of graph or SOM clustering
- **Shared normalization** - Compatible with the existing SPACEMAP phenotyping notebooks
- **Configurable marker mode** - Corrected markers by default, raw markers optional
- **Interactive annotation** - Uses the same phenotype assignment studio as the other methods

In [ ]:
# Imports & Environment Setup
import os, sys, warnings, logging, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

project_root = Path().resolve().parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from spacemap.config import default_config
from spacemap.ph_vortex_utils import (
    load_or_compute_normalized,
    characterize_clusters,
    create_cluster_heatmap,
    launch_phenotype_assignment_app,
    apply_phenotypes,
    evaluate_vortex_k_grid,
    recommend_vortex_k,
    create_vortex_k_diagnostic_plot,
    create_vortex_qc_plots,
    intelligent_vortex_clustering,
)

try:
    import scanpy as sc
    print(f'Scanpy {sc.__version__} available')
except Exception:
    sc = None
    print('Scanpy not available; UMAP generation may be skipped.')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('step6e_vortex')

print(f'Project root: {project_root}')
print('Step 6E: Vortex Phenotyping Setup Complete')

## Configuration

Configure the Vortex/X-shift clustering parameters below. The main decision points are marker mode, `k` selection, sketch size, and QC output behavior.

In [ ]:
# =============================================================================

# CONFIGURATION

# =============================================================================


INPUT_DIR = project_root / 'outputs/batch_corrected'
OUTPUT_DIR = default_config.output_dir / 'phenotyping'
PLOTS_DIR = OUTPUT_DIR / 'vortex/plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

#input_file = INPUT_DIR / 'all_samples_combined_corrected.csv'
input_file = Path(f'{DATA_DIR}/example_project/outputs/annotations/csv/corrected/INNATE_4_Images.csv')

USE_CORRECTED_MARKERS = True
NORMALIZATION_FLAGS = dict(z1=True, z2=False, cdf=False, nlog=False)
NORMALIZATION_GROUP_COLS = ['batch_id']

FILTER_CRITERIA = {
    'tissue': 1,
    # 'corrected': 1,
}
LOW_EXPRESSION_Z_THRESHOLD = -0.75
CELL_ID_COLUMN = 'cell_id'

CORR_MARKERS = [
    'CD31', 'E-cadherin', 'CD20', 'CD4', 'Pan-Cytokeratin', 'CD68',
    'Podoplanin', 'CD11c', 'CD8', 'HLA-DR', 'CD3e', 'CD45', 'CD15',
    'FoxP3', 'PDGFRa', 'aSMA', 'CD163', 'FN1', 'S100A9', 'CD11b',
    'CD38', 'Vimentin'
]
NON_CORR_MARKERS = [

]

MARKERS_BASE = CORR_MARKERS + NON_CORR_MARKERS
CLUSTERING_MARKERS = (
    [f'{marker}_corr' for marker in CORR_MARKERS] if USE_CORRECTED_MARKERS else CORR_MARKERS
) + NON_CORR_MARKERS

AUTO_SELECT_K = True
VORTEX_K = 35
K_CANDIDATES = [10, 15, 20, 25, 30, 35, 45, 60]
VORTEX_N_CLUSTERS_TARGET = 18
K_SCAN_SKETCH_SIZE = 25000
SUBSAMPLE_SIZE = 100000
SUBSAMPLE_METHOD = 'geosketch'
KNN_NEIGHBORS = 15
MIN_CLUSTER_SIZE = 20
GENERATE_UMAP = True
UMAP_N_NEIGHBORS = 15
MAX_PLOT_CELLS = 250000
RANDOM_SEED = 42
marker_mode = 'corrected' if USE_CORRECTED_MARKERS else 'raw'


print('Configuration loaded:')
print(f' Input file: {input_file}')
print(f' Plot output: {PLOTS_DIR}')
print(f' Cell ID column: {CELL_ID_COLUMN}')
print(f' Marker mode: {marker_mode}')
print(f' Filter criteria: {FILTER_CRITERIA}')
print(f' Low-expression threshold: mean marker z-score > {LOW_EXPRESSION_Z_THRESHOLD}')
print(f' Total configured markers: {len(CLUSTERING_MARKERS)}')
print(f' Auto-select k: {AUTO_SELECT_K}')
print(f' Default/manual k: {VORTEX_K}')
print(f' K candidates: {K_CANDIDATES}')
print(f' Subsample size: {SUBSAMPLE_SIZE:,}')
print(f' Subsample method: {SUBSAMPLE_METHOD}')


## Step 1: Data Loading and Preparation

Load the input data, apply optional filtering, compute shared normalization, and prepare the Vortex clustering matrix.

In [ ]:
# =============================================================================
# STEP 1: DATA LOADING & NORMALIZATION
# =============================================================================
print('Step 1: Loading data and preparing Vortex input...')

if not input_file.exists():
    raise FileNotFoundError(f'Input file not found: {input_file}')

data = pd.read_csv(input_file, low_memory=False)
print(f'Loaded {len(data):,} rows')

for column, value in FILTER_CRITERIA.items():
    if value is not None:
        if column not in data.columns:
            print(f"Warning: '{column}' column not found - skipping filter")
            continue
        before = len(data)
        data = data[data[column] == value].copy()
        print(f' Applied filter {column} == {value}: {len(data):,}/{before:,} rows kept')

if CELL_ID_COLUMN not in data.columns:
    raise ValueError(f"Critical: '{CELL_ID_COLUMN}' column missing from input data")

available_markers = [marker for marker in CLUSTERING_MARKERS if marker in data.columns]
missing_markers = [marker for marker in CLUSTERING_MARKERS if marker not in data.columns]
if not available_markers:
    raise ValueError('No requested markers were found in the input file')

print(f'Available clustering markers: {len(available_markers)}')
if missing_markers:
    print(f'Missing markers skipped: {missing_markers[:10]}')

CACHE_DIR = OUTPUT_DIR / 'normalized_cache'
cache_name = 'normalized_markers.csv' if USE_CORRECTED_MARKERS else 'normalized_markers_raw.csv'
cache_csv = CACHE_DIR / cache_name
print(f'Using normalization cache: {cache_csv}')

norm_df = load_or_compute_normalized(
    df=data[[CELL_ID_COLUMN] + NORMALIZATION_GROUP_COLS + available_markers],
    marker_cols=available_markers,
    group_cols=NORMALIZATION_GROUP_COLS,
    suffix='_norm',
    normalization_flags=NORMALIZATION_FLAGS,
    cache_path=cache_csv,
    id_col=CELL_ID_COLUMN,
)

norm_cols = [column for column in norm_df.columns if column.endswith('_norm')]
print(f'Normalized columns ready: {len(norm_cols)}')

if CELL_ID_COLUMN in norm_df.columns:
    data = data.merge(norm_df, on=CELL_ID_COLUMN, how='left')
else:
    norm_df = norm_df.reindex(data.index)
    data = pd.concat([data, norm_df], axis=1)

pre_filter_count = len(data)
low_expression_scores = data[norm_cols].fillna(0).mean(axis=1)
data = data[low_expression_scores > LOW_EXPRESSION_Z_THRESHOLD].copy()
filtered_count = pre_filter_count - len(data)

print('Filtering results:')
print(f' Before expression filter: {pre_filter_count:,}')
print(f' After expression filter:  {len(data):,}')
print(f' Removed:                  {filtered_count:,} ({filtered_count / pre_filter_count * 100:.2f}%)')
print(f' Mean marker z-score cutoff: > {LOW_EXPRESSION_Z_THRESHOLD}')

X = data[norm_cols].fillna(0).astype(float)
print(f'Prepared clustering matrix: {X.shape[0]:,} cells x {X.shape[1]} markers')

## Step 2: Vortex Clustering

Run the Vortex/X-shift clustering workflow. This step can optionally scan multiple `k` values on a sketch, save a diagnostic figure, and then run the final clustering with the chosen `k`.

In [ ]:
# =============================================================================

# STEP 2: VORTEX CLUSTERING

# =============================================================================

from IPython.display import Image, display
print('Step 2: Running Vortex clustering...')
print(f'Dataset: {len(data):,} cells x {len(norm_cols)} markers')

selected_k = VORTEX_K
k_results = None
k_plot_path = None
if AUTO_SELECT_K:
    print('Running Vortex K diagnostic scan...')
    try:
        k_results = evaluate_vortex_k_grid(
            df=data,
            norm_cols=norm_cols,
            k_values=K_CANDIDATES,
            sketch_size=K_SCAN_SKETCH_SIZE,
            subsample_method=SUBSAMPLE_METHOD,
            min_cluster_size=MIN_CLUSTER_SIZE,
            random_seed=RANDOM_SEED,
            logger=logger,
        )
        k_results_path = OUTPUT_DIR / 'vortex_k_scan_results.csv'
        k_results.to_csv(k_results_path, index=False)
        print(f'K diagnostic table saved: {k_results_path}')
        display(k_results)
        k_plot_path = create_vortex_k_diagnostic_plot(
            results_df=k_results,
            plots_dir=PLOTS_DIR,
            filename='vortex_k_diagnostics.png',
        )
        print(f'K diagnostic plot saved: {k_plot_path}')
        display(Image(filename=str(k_plot_path)))
        recommended = recommend_vortex_k(
            results_df=k_results,
            target_n_clusters=VORTEX_N_CLUSTERS_TARGET,
        )
        selected_k = int(recommended['k_neighbors'])
        print('Recommended Vortex k based on diagnostic scan:')
        print(recommended)
    except Exception as exc:
        print(f'K diagnostic scan failed, falling back to manual k={VORTEX_K}: {exc}')
        selected_k = VORTEX_K
else:
    print(f'Using manual Vortex k without scan: {selected_k}')

vortex_params = dict(
    k_neighbors=selected_k,
    subsample_size=SUBSAMPLE_SIZE,
    subsample_method=SUBSAMPLE_METHOD,
    random_seed=RANDOM_SEED,
    knn_neighbors=KNN_NEIGHBORS,
    min_cluster_size=MIN_CLUSTER_SIZE,
    generate_umap=GENERATE_UMAP,
    umap_neighbors=UMAP_N_NEIGHBORS,
)


cluster_assignments, umap_coords, vortex_diagnostics = intelligent_vortex_clustering(
    df=data,
    norm_cols=norm_cols,
    vortex_params=vortex_params,
    logger=logger,
)

data['vortex_cluster'] = cluster_assignments
if np.isfinite(umap_coords).any():
    data['umap_x'] = umap_coords[:, 0]
    data['umap_y'] = umap_coords[:, 1]
    print('UMAP coordinates added to data frame')
else:

    print('UMAP coordinates not available for this run')



cluster_sizes = data['vortex_cluster'].value_counts().sort_index()

final_cluster_count = data['vortex_cluster'].nunique()

runtime_seconds = vortex_diagnostics['runtime_seconds']



print('Vortex clustering complete:')
print(f' Selected k:        {selected_k}')
print(f' Final clusters:    {final_cluster_count}')
print(f' Min cluster size:  {cluster_sizes.min()}')
print(f' Median cluster:    {cluster_sizes.median()}')
print(f' Max cluster size:  {cluster_sizes.max()}')
print(f' Runtime (seconds): {runtime_seconds}')


## Step 3: Cluster Characterization

Compute per-cluster marker-expression summaries and cell counts to support phenotype interpretation.

In [ ]:
# =============================================================================
# STEP 3: CLUSTER CHARACTERIZATION
# =============================================================================
print('Step 3: Characterizing Vortex clusters...')

char_df = characterize_clusters(
    data=data,
    cluster_col='vortex_cluster',
    marker_cols=norm_cols,
)

char_path = OUTPUT_DIR / 'vortex_cluster_characterization.csv'
char_df.to_csv(char_path, index=False)

print(f'Cluster characterization saved: {char_path}')
print(f'Characterized {len(char_df)} clusters with {len(norm_cols)} normalized markers')
display(char_df.head())

## Step 4: Expression Heatmap

Generate a clustered heatmap of Vortex cluster expression patterns to guide phenotype assignment.

In [ ]:
# =============================================================================
# STEP 4: EXPRESSION HEATMAP
# =============================================================================
from IPython.display import Image, display

print('Step 4: Creating Vortex expression heatmap...')

reordered_cluster_ids, heatmap_file = create_cluster_heatmap(
    char_df=char_df,
    plots_dir=PLOTS_DIR,
    cluster_col='vortex_cluster',
    title='Vortex Cluster Expression Heatmap',
    filename='vortex_cluster_expression_heatmap.png',
)

print(f'Expression heatmap saved: {heatmap_file}')
print(f'Cluster ordering: {reordered_cluster_ids}')
display(Image(filename=str(heatmap_file)))

## Step 5: Interactive Phenotype Assignment

Assign biological phenotypes to Vortex clusters using the standard SPACEMAP phenotype assignment studio.

**Workflow**: Use the GUI to assign phenotypes from the Step 4 heatmap ordering, or skip to the scripted assignment cell below.

In [ ]:
# =============================================================================
# STEP 5A: INTERACTIVE PHENOTYPE ASSIGNMENT
# =============================================================================
print('Step 5: Launching interactive phenotype assignment studio...')

if 'VORTEX_PHENOTYPES' not in globals():
    VORTEX_PHENOTYPES = {}

VORTEX_PHENOTYPES = launch_phenotype_assignment_app(
    char_df=char_df,
    reordered_cluster_ids=reordered_cluster_ids,
    meta_cluster_phenotypes=VORTEX_PHENOTYPES,
)

print(f'Phenotype assignments captured: {len(VORTEX_PHENOTYPES)} clusters')
print(f'Assigned clusters: {sorted(VORTEX_PHENOTYPES.keys())}')

## Step 5B: Manual Phenotype Assignment

Optional scripted phenotype mapping for reproducible runs after reviewing the Vortex heatmap and QC figures.

In [ ]:
# =============================================================================
# STEP 5B: OPTIONAL MANUAL PHENOTYPE ASSIGNMENT
# =============================================================================
# Example mapping structure:
# example_pheno_map = {
#     'T-Cells': [0, 1, 5],
#     'B-Cells': [2, 8],
# }
#
# if 'VORTEX_PHENOTYPES' not in globals():
#     VORTEX_PHENOTYPES = {}
#
# for phenotype, clusters in example_pheno_map.items():
#     for cluster_id in clusters:
#         VORTEX_PHENOTYPES[cluster_id] = phenotype
#
# print(f'Total assigned (including scripted): {len(VORTEX_PHENOTYPES)}')

## Step 6: UMAP Visualization and QC

Generate a compact QC dashboard including cluster-size distributions and UMAP summaries. If phenotype assignments exist, the QC plot will also show phenotype-colored UMAPs.

In [ ]:
# =============================================================================
# STEP 6: UMAP VISUALIZATION AND QC
# =============================================================================
from IPython.display import Image, display

print('Step 6: Generating Vortex QC visualizations...')

if 'VORTEX_PHENOTYPES' in globals() and VORTEX_PHENOTYPES:
    data['phenotype_vortex'] = data['vortex_cluster'].map(VORTEX_PHENOTYPES).fillna('Unassigned')
    print(f"Phenotype assignments mapped to {(data['phenotype_vortex'] != 'Unassigned').sum():,} cells")
else:
    data['phenotype_vortex'] = 'Unassigned'
    print("No phenotype assignments found. Using 'Unassigned' for all cells.")

qc_dashboard_path = create_vortex_qc_plots(
    data=data,
    plots_dir=PLOTS_DIR,
    cluster_col='vortex_cluster',
    phenotype_col='phenotype_vortex',
    max_plot_cells=MAX_PLOT_CELLS,
    random_seed=RANDOM_SEED,
    filename='vortex_qc_dashboard.png',
)

print(f'QC dashboard saved: {qc_dashboard_path}')
display(Image(filename=str(qc_dashboard_path)))

phenotype_counts = data['phenotype_vortex'].value_counts()
print('Phenotype distribution:')
for phenotype, count in phenotype_counts.items():
    print(f'  {phenotype}: {count:,} cells ({count / len(data) * 100:.1f}%)')

## Step 7: Apply Phenotypes and Save Results

Apply the phenotype mapping to the full dataset and update the shared unified phenotyping output.

In [ ]:
# =============================================================================
# STEP 7: APPLY PHENOTYPES AND SAVE RESULTS
# =============================================================================
print('Step 7: Applying Vortex phenotypes and saving results...')

if 'VORTEX_PHENOTYPES' not in globals() or not VORTEX_PHENOTYPES:
    print('No phenotype assignments available. Please run Step 5A or Step 5B first.')
else:
    clean_data = data[[column for column in data.columns if not column.endswith('_norm')]].copy()
    result = apply_phenotypes(
        data=clean_data,
        phenotype_map=VORTEX_PHENOTYPES,
        output_base_dir=OUTPUT_DIR,
        cell_id_column=CELL_ID_COLUMN,
        cluster_col='vortex_cluster',
        method_prefix='vortex',
    )

    print('Results saved successfully:')
    print(f" Main dataset:       {result['unified_file']}")
    print(f" Summary statistics: {result['summary_file']}")
    print(f" Mapping file:       {result['mapping_file']}")
    display(result['summary_df'].head(10))

## Step 8: Export Per-Batch CSVs (Optional)

Export batch-specific CSV files from the unified Vortex phenotyping output for downstream sample-level analysis.

In [ ]:
# =============================================================================
# STEP 8: OPTIONAL BATCH EXPORT
# =============================================================================
import re

EXPORT_ALL_BATCHES = False
EXPORT_PREFIXES = ['P11_roi_1']
EXPORT_SUBDIR = OUTPUT_DIR / 'vortex_batch_exports'
EXPORT_SUBDIR.mkdir(parents=True, exist_ok=True)

UNIFIED_FILE = OUTPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'
BATCH_COL = 'batch_id'
PHENOTYPE_COL = 'phenotype_vortex'

if 'result' in globals() and isinstance(result, dict) and 'unified_file' in result:
    unified_path = Path(result['unified_file'])
else:
    unified_path = UNIFIED_FILE

print('Step 8: Optional batch export...')
if not unified_path.exists():
    raise FileNotFoundError(f'Unified phenotyping file not found: {unified_path}. Run Step 7 first.')

unified_df = pd.read_csv(unified_path)
print(f'Loaded unified phenotyping file: {unified_path} ({len(unified_df):,} rows)')

if BATCH_COL not in unified_df.columns:
    print(f"Column '{BATCH_COL}' not found; skipping export.")
else:
    batches = unified_df[BATCH_COL].dropna().astype(str).unique().tolist()
    if EXPORT_ALL_BATCHES:
        target_batches = batches
        print(f'Exporting all {len(target_batches)} batches...')
    else:
        target_batches = []
        for prefix in EXPORT_PREFIXES:
            matched = [batch for batch in batches if batch.startswith(prefix)]
            target_batches.extend(matched)
        seen = set()
        target_batches = [batch for batch in target_batches if not (batch in seen or seen.add(batch))]
        print(f'Exporting {len(target_batches)} batches matching prefixes {EXPORT_PREFIXES}')

    if not target_batches:
        print('No batches match the export criteria.')
    else:
        for batch in target_batches:
            df_batch = unified_df[unified_df[BATCH_COL].astype(str) == batch].copy()
            safe_name = re.sub(r'[^A-Za-z0-9._-]+', '_', batch)
            out_path = EXPORT_SUBDIR / f'{safe_name}_phenotype_vortex.csv'
            df_batch.to_csv(out_path, index=False)
        print(f'Export complete. Files saved under: {EXPORT_SUBDIR}')

## Step 9: Cleanup Normalized Cache

Delete the shared normalization cache file so the next run can recompute normalization from scratch when needed.

In [ ]:
# =============================================================================
# STEP 9: CLEANUP NORMALIZED CACHE
# =============================================================================
print('Step 9: Cleaning up normalized cache...')

if 'cache_csv' in globals() and cache_csv.exists():
    cache_csv.unlink()
    print(f'Deleted normalized cache: {cache_csv}')

    cache_dir = cache_csv.parent
    if cache_dir.exists() and not any(cache_dir.iterdir()):
        cache_dir.rmdir()
        print(f'Removed empty cache directory: {cache_dir}')
else:
    print('No cache file found to clean up')

print('Cache cleanup complete.')

## Summary

The Vortex/X-shift phenotyping workflow is now available as a separate SPACEMAP notebook with the same downstream structure as the existing Leiden and SOM methods.

**Generated outputs** include Vortex cluster assignments, cluster characterization tables, phenotype mappings, K-selection diagnostics, heatmaps, and QC figures.

**Next steps**: compare Vortex-derived phenotypes with Leiden and SOM outputs, inspect the QC dashboard for cluster stability, and proceed to downstream spatial analysis using the unified phenotyped CSV.